In [167]:
import struct

def read_variable_length(data, offset):
    # reads a variable acording to the midi the bottom 7 bits are used for value
    # if the 8th bit is 0 the value is done if the 8th bit is 1 grabs one more byte
    value = 0
    while True:
        byte = data[offset]
        offset += 1
        value = (value << 7) | (byte & 0x7F)
        if not (byte & 0x80):
            break
    return value, offset
    
def readMidiFile(path):
    src = open(path, "rb")
    current_ticks = 0
    tempo = 500000
    ticks_per_quarter = 480
    header_dict = {}
    while True:
        chunkID = src.read(4)
        chunklen = int.from_bytes(src.read(4))
        chunk = src.read(chunklen)
        delta_times = False
        if chunkID == b'MThd':
            header_dict['format'], header_dict['ntracks'], header_dict['tickdiv']  = struct.unpack('>HHH', chunk[0:6])
            header_dict['µs_per_tick'] = tempo/header_dict['tickdiv'] 
            if header_dict['format'] != 1:
                    print('Only format 1 MIDI files are suported')
            print('Header: ', header_dict)
        elif chunkID == b'MTrk':
            offset = 0
            print("track:")
            #print(chunk)
            while offset < chunklen:
                # Read delta time
                delta, offset = read_variable_length(chunk, offset)
                current_ticks += delta
                event_status = chunk[offset]
                offset += 1
                if event_status == 0xFF: #Meta event
                    meta_type = chunk[offset]
                    offset += 1
                    meta_length = chunk[offset]
                    offset += 1
                    meta_data = chunk[offset:offset+meta_length]
                    offset += meta_length
                    if meta_type == 0x2F and meta_data == b'':
                        print(current_ticks,"- Metadata > End of track")
                    elif meta_type == 0x03:
                        print(current_ticks,"- Metadata > Track/Sequence name > ",meta_data)
                    elif meta_type == 0x51:
                        tempo = struct.unpack('>i', b'\x00'+meta_data)[0]
                        header_dict['µs_per_tick'] = tempo/header_dict['tickdiv']
                        print(current_ticks,"- Metadata > Tempo > ",tempo)
                    elif meta_type == 0x58:
                        num, denom, clocks_per_clck, notes_per_beat = struct.unpack('>bbbb', meta_data)
                        print(current_ticks,"- Metadata > Time signature >",num, denom, clocks_per_clck, notes_per_beat)
                    else:
                        print(current_ticks,'- type - ',meta_type,' - Meta Data > ',meta_data)
                elif event_status//16 == 9 or event_status//16 == 8:
                    note = chunk[offset]
                    offset += 1
                    vel = chunk[offset]
                    offset += 1
                    if event_status//16 == 9:
                        note_ac = 'Note On'
                    else:
                        note_ac = 'Note Off'
                    print(note_ac+' > n:',note,' v:',vel,'t:',int(current_ticks*header_dict['µs_per_tick']))
                elif event_status//16 == 11:
                    controller = chunk[offset]
                    offset += 1
                    value = chunk[offset]
                    offset += 1
                    print(current_ticks,' - controller > ctrl: ',controller,'val: ',value)
                elif event_status//16 == 12:
                    pgrm_n = chunk[offset]
                    offset += 1
                    print(current_ticks,' - program > pgrm_n: ',pgrm_n)
                elif event_status//16 == 14:
                    lsb = chunk[offset]
                    offset += 1
                    msb = chunk[offset]
                    offset += 1
                    print(current_ticks,' - pitchBend > lsb: ',lsb,' msb: ',msb)
                else:
                    print(current_ticks,' - ',hex(event_status))
        else:
            break

In [168]:
mid = readMidiFile('../ChipTune Melodies/Kid Dracula - Go-Go at the Great Castle.mid')

Header:  {'format': 1, 'ntracks': 2, 'tickdiv': 480, 'µs_per_tick': 1041.6666666666667}
track:
0 - Metadata > Track/Sequence name >  b''
0 - Metadata > Time signature > 4 2 24 8
0 - Metadata > Tempo >  535714
0 - Metadata > End of track
track:
0  - controller > ctrl:  121 val:  0
0 - Metadata > Track/Sequence name >  b'main track'
0  - controller > ctrl:  10 val:  64
0  - controller > ctrl:  7 val:  100
0  - controller > ctrl:  11 val:  127
0  - controller > ctrl:  101 val:  0
0  - controller > ctrl:  100 val:  2
0  - controller > ctrl:  6 val:  64
0  - controller > ctrl:  101 val:  0
0  - controller > ctrl:  100 val:  1
0  - controller > ctrl:  6 val:  64
0  - controller > ctrl:  38 val:  0
0  - controller > ctrl:  101 val:  0
0  - controller > ctrl:  100 val:  0
0  - controller > ctrl:  6 val:  12
0  - pitchBend > lsb:  0  msb:  64
0  - controller > ctrl:  1 val:  0
0  - program > pgrm_n:  80
Note On > n: 85  v: 100 t: 803571
Note Off > n: 85  v: 0 t: 1071428
Note On > n: 83  v: 100 

In [68]:
from mido import MidiFile

# Load the MIDI file
mid = MidiFile('../ChipTune Melodies/Kid Dracula - Go-Go at the Great Castle.mid')

# Print file metadata
print(f"Type: {mid.type}, Ticks per beat: {mid.ticks_per_beat}")

# Iterate through tracks and messages
for track in mid.tracks:
    for msg in track:
        print(msg)

Type: 1, Ticks per beat: 480
MetaMessage('track_name', name='', time=0)
MetaMessage('time_signature', numerator=4, denominator=4, clocks_per_click=24, notated_32nd_notes_per_beat=8, time=0)
MetaMessage('set_tempo', tempo=535714, time=0)
MetaMessage('end_of_track', time=0)
control_change channel=0 control=121 value=0 time=0
MetaMessage('track_name', name='main track', time=0)
control_change channel=0 control=10 value=64 time=0
control_change channel=0 control=7 value=100 time=0
control_change channel=0 control=11 value=127 time=0
control_change channel=0 control=101 value=0 time=0
control_change channel=0 control=100 value=2 time=0
control_change channel=0 control=6 value=64 time=0
control_change channel=0 control=101 value=0 time=0
control_change channel=0 control=100 value=1 time=0
control_change channel=0 control=6 value=64 time=0
control_change channel=0 control=38 value=0 time=0
control_change channel=0 control=101 value=0 time=0
control_change channel=0 control=100 value=0 time=0


In [116]:
b'0'

b'0'

In [117]:
bytearray(0x00)

bytearray(b'')

In [134]:
0x91//16

9